# IE7615 - Milestone 2 - Photo Generation

This notebook will build 640x640 scenes from the cleaned single-object photos. The code that we have written will randomly select 2-5 object photos from the dataset and will place them in random spots inside of the 640x640 canvas. Extra car has been given to ensure that no photos will overlap and has also built in a functionality to skip placement when a certain amount of tries has been reached. Additionally, the scenes that these object photos will be placed into are the background photos from the data set as well. This should be better than a blank canvas to force the model to learn where objects are instead of generalizing to the only spots on the canvas that aren't blank. Also each of the scenes that are generated will be with respect to the train/validation/test split so that no photos end up in another splits groups. It will also write YOLO labels and save the dataset to the shared Drive folder that our group is using under the name "scenes_v1.tar". We will be running this all on colab as well. 

In [ ]:
# Set up colab
from google.colab import drive
drive.mount("/content/drive")

!if [ -d /content/repo ]; then git -C /content/repo pull; else git clone https://github.com/connor-rivera/ie7615-group2-object-detection.git /content/repo; fi
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
# Import all libraries and load photos
import os
import random
import tarfile
import yaml
from tqdm import tqdm
from src import config
from src.data import unpack_cleaned, load_photos, split_pools
from src.compose import make_scene, yolo_lines

image_dir = unpack_cleaned(config.CLEANED_TAR, config.LOCAL)
photos = load_photos("records/manifest_v1.csv", image_dir)
print(photos.groupby(["split", "is_background"]).size())

In [ ]:
# Building the scenes
name = "scenes_v1"
out = os.path.join(config.LOCAL, name)
rng = random.Random(config.seed) # One generator for the whole run so the dataset can be repliocated exactly

for split, n_scenes in config.scenes.items():
    backgrounds, objects = split_pools(photos, split)
    bg_paths = backgrounds["path"].tolist()
    os.makedirs(f"{out}/images/{split}", exist_ok = True)
    os.makedirs(f"{out}/labels/{split}", exist_ok = True)
    for k in tqdm(range(n_scenes), desc = split):
        scene, labels = make_scene(rng.choice(bg_paths), objects, rng)
        stem = f"{split}_{k:04d}"
        scene.save(f"{out}/images/{split}/{stem}.jpg", quality = 95)
        with open(f"{out}/labels/{split}/{stem}.txt", "w") as f:
            f.write("\n".join(yolo_lines(labels)) + "\n")
        

In [ ]:
# writing the data.yaml
data = {
    "path": out,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: f"OBJ{i + 1:03d}" for i in range(73)}, # Logic where classes begin counts at 0, YOLO class 0 is OBJ001
}

with open(f"{out}/data.yaml", "w") as f:
    yaml.safe_dump(data, f, sort_keys = False)
print(open(f"{out}/data.yaml").read() [:300])

In [ ]:
# Save things to the Drive
with tarfile.open(os.path.join(config.OUTPUTS, f"{name}.tar"), "w") as tar:
    tar.add(out, arcname= name)
print("saved", os.path.join(config.OUTPUTS, f"{name}.tar"))

In [ ]:
# Sample figure to make sure things are working
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

fig, axes = plt.subplots(2, 3, figsize = (15, 10))
for ax, k in zip(axes.flat, range(6)):
    stem = f"train_{k:04d}"
    img = Image.open(f"{out}/images/train/{stem}.jpg")
    draw = ImageDraw.Draw(img)
    for line in open(f"{out}/labels/train/{stem}.txt").read().splitlines():
        cls, cx, cy, w, h = line.split()
        cx, cy, w, h = [float(v) * config.canvas for v in (cx, cy, w, h)]
        draw.rectangle([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], outline = "lime", width = 4)
        draw.text((cx - w / 2 + 6, cy - h / 2 + 6), f"OBJ{int(cls) + 1:03d}", fill = "lime")
    ax.imshow(img)
    ax.set_title(stem)
    ax.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(config.OUTPUTS, "sample_scenes.png"), dpi = 100)
plt.show()